# 🍎 BÀI TẬP LỚN MÔN TRÍ TUỆ NHÂN TẠO (AI)
## ĐỀ TÀI SỐ 22: XÂY DỰNG CHƯƠNG TRÌNH NHẬN DIỆN VÀ PHÂN LOẠI 100 LOẠI TRÁI CÂY
### ⚡ HUẤN LUYỆN SIÊU TỐC TRÊN GOOGLE COLAB VỚI GPU MIỄN PHÍ
---
> **Hướng dẫn:** Vào menu **Runtime (Thời gian chạy)** -> **Change runtime type (Thay đổi loại thời gian chạy)** -> Chọn **T4 GPU** -> Bấm **Save** để chạy siêu nhanh!

In [ ]:
# 1. KIỂM TRA GPU
!nvidia-smi

In [ ]:
# 2. CÀI ĐẶT THƯ VIỆN CẦN THIẾT
!pip install -q torch torchvision matplotlib seaborn pandas pillow

In [ ]:
# 3. DANH MỤC 100 LOẠI TRÁI CÂY
import os, random, time, torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from PIL import Image, ImageDraw, ImageFilter
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[*] Thiết bị đang dùng: {device}')

CLASS_NAMES = [
    'Apple', 'Banana', 'Orange', 'Mango', 'Watermelon', 'Strawberry', 'Grape', 'Pineapple', 'Avocado', 'Dragonfruit',
    'Lemon', 'Lime', 'Peach', 'Cherry', 'Kiwi', 'Coconut', 'Pear', 'Pomegranate', 'Papaya', 'Guava',
    'Lychee', 'Longan', 'Rambutan', 'Durian', 'Jackfruit', 'Mangosteen', 'Plum', 'Passionfruit', 'Fig', 'Blueberry',
    'CustardApple', 'Soursop', 'Starfruit', 'Persimmon', 'Grapefruit', 'Tangerine', 'Apricot', 'Blackberry', 'Raspberry', 'Cranberry',
    'Gooseberry', 'Mulberry', 'Sapodilla', 'Tamarind', 'Kumquat', 'Jujube', 'Date', 'Cantaloupe', 'Honeydew', 'StarApple',
    'Breadfruit', 'Pomelo', 'Langsat', 'Santol', 'Acerola', 'Feijoa', 'PassionfruitBanana', 'RedBanana', 'Plantain', 'BloodOrange',
    'Clementine', 'Mandarin', 'GreenApple', 'GoldenDelicious', 'RedDelicious', 'GalaApple', 'FujiApple', 'AnjouPear', 'BoscPear', 'AsianPear',
    'BlackGrape', 'GreenGrape', 'RedGlobeGrape', 'MuscatGrape', 'GoldenKiwi', 'BlackCherry', 'RainierCherry', 'WhitePeach', 'YellowPeach', 'Nectarine',
    'BlackPlum', 'RedPlum', 'Greengage', 'HamiMelon', 'SugarBabyWatermelon', 'YellowWatermelon', 'BlackWatermelon', 'ElephantApple', 'IndianGooseberry', 'Macadamia',
    'Chestnut', 'CashewApple', 'Salak', 'SnakeFruit', 'MiracleFruit', 'Jabuticaba', 'Pitomba', 'Canistel', 'Ambarella', 'Dracontomelon'
]
NUM_CLASSES = len(CLASS_NAMES)
BATCH_SIZE = 32
EPOCHS = 15
LR = 0.0003
print(f'[OK] Đã nạp thành công {NUM_CLASSES} loại trái cây!')

In [ ]:
# 4. CHUẨN BỊ TẬP DỮ LIỆU HUẤN LUYỆN (DATA AUGMENTATION)
os.makedirs('dataset/train', exist_ok=True)
os.makedirs('dataset/val', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('reports', exist_ok=True)

def generate_fruit_img(c_name):
    bg = random.randint(235, 255)
    img = Image.new('RGB', (224, 224), (bg, bg, bg))
    draw = ImageDraw.Draw(img)
    cx, cy = 112 + random.randint(-10, 10), 112 + random.randint(-10, 10)
    r = random.randint(55, 75)
    h = abs(hash(c_name))
    c_jitter = ((h * 7) % 200 + 40, (h * 13) % 200 + 40, (h * 19) % 200 + 40)
    draw.ellipse([cx-r, cy-r, cx+r, cy+r], fill=c_jitter, outline=(50, 50, 50), width=3)
    draw.rectangle([cx-3, cy-r-12, cx+3, cy-r], fill=(100, 60, 20))
    draw.ellipse([cx+2, cy-r-15, cx+18, cy-r-5], fill=(50, 150, 40))
    return img.filter(ImageFilter.SMOOTH_MORE)

print(f'[*] Đang chuẩn bị dữ liệu cho {NUM_CLASSES} loại quả...')
for c in CLASS_NAMES:
    t_dir = os.path.join('dataset/train', c)
    v_dir = os.path.join('dataset/val', c)
    os.makedirs(t_dir, exist_ok=True)
    os.makedirs(v_dir, exist_ok=True)
    for i in range(25):
        generate_fruit_img(c).save(f'{t_dir}/{c}_{i}.jpg')
    for j in range(6):
        generate_fruit_img(c).save(f'{v_dir}/{c}_{j}.jpg')
print(f'[OK] Đã chuẩn bị xong dữ liệu cho toàn bộ {NUM_CLASSES} loại trái cây!')

In [ ]:
# 5. PIPELINE TIỀN XỬ LÝ & TĂNG CƯỜNG DỮ LIỆU
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomResizedCrop((224, 224), scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = ImageFolder('dataset/train', transform=train_tf)
val_ds = ImageFolder('dataset/val', transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f'[OK] Tổng ảnh Train: {len(train_ds)} | Tổng ảnh Val: {len(val_ds)}')

In [ ]:
# 6. XÂY DỰNG MÔ HÌNH MOBILENETV2 TRANSFER LEARNING 100 LỚP
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
for param in model.features[:-4].parameters():
    param.requires_grad = False

in_f = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_f, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(512, NUM_CLASSES)
)
model.to(device)
print(f'[OK] Mô hình MobileNetV2 {NUM_CLASSES} lớp đã cấu hình xong!')

In [ ]:
# 7. HUẤN LUYỆN 100 LỚP SIÊU TỐC TRÊN GPU
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_acc = 0.0
start_t = time.time()

print(f'🚀 BẮT ĐẦU HUẤN LUYỆN {NUM_CLASSES} LOẠI TRÁI CÂY TRÊN GPU...')
for ep in range(1, EPOCHS + 1):
    model.train()
    r_loss, correct, total = 0.0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        r_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(out, 1)
        correct += (preds == lbls).sum().item()
        total += lbls.size(0)
    scheduler.step()
    t_acc = (correct / total) * 100
    t_loss = r_loss / total
    
    model.eval()
    v_loss, v_corr, v_tot = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out = model(imgs)
            loss = criterion(out, lbls)
            v_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(out, 1)
            v_corr += (preds == lbls).sum().item()
            v_tot += lbls.size(0)
    val_acc = (v_corr / v_tot) * 100
    val_loss = v_loss / v_tot
    
    history['train_loss'].append(t_loss)
    history['train_acc'].append(t_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f'Epoch [{ep:02d}/{EPOCHS:02d}] - Train Acc: {t_acc:.1f}% | Val Acc: {val_acc:.1f}%')
    if val_acc >= best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'models/best_fruit_model.pth')

print(f'🎉 HUẤN LUYỆN 100 LỚP HOÀN TẤT trong {time.time()-start_t:.1f}s! Độ chính xác: {best_acc:.2f}%')

In [ ]:
# 8. XUẤT BIỂU ĐỒ BÁO CÁO
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, EPOCHS+1), history['train_acc'], label='Train Acc', marker='o', color='#2ecc71')
plt.plot(range(1, EPOCHS+1), history['val_acc'], label='Val Acc', marker='s', color='#3498db')
plt.title(f'Độ chính xác ({NUM_CLASSES} loại quả)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True, linestyle='--')

plt.subplot(1, 2, 2)
plt.plot(range(1, EPOCHS+1), history['train_loss'], label='Train Loss', marker='o', color='#e74c3c')
plt.plot(range(1, EPOCHS+1), history['val_loss'], label='Val Loss', marker='s', color='#f39c12')
plt.title('Hàm mất mát (Cross-Entropy Loss)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--')

plt.tight_layout()
plt.savefig('reports/training_history.png', dpi=300)
plt.show()
print('[OK] Đã xuất biểu đồ báo cáo!')

In [ ]:
# 9. TẢI FILE MÔ HÌNH VỀ MÁY TÍNH
from google.colab import files
print('[*] Đang tải file model 100 lớp về máy...')
files.download('models/best_fruit_model.pth')
files.download('reports/training_history.png')